In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd
from sklearn.linear_model import LinearRegression
import shap
from time import perf_counter

/Users/maxi/miniconda3/envs/EML/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Load arbitrary data

In [2]:
data = pd.read_parquet("../data/california_housing_prices.parquet")
X = data.drop(columns="HousePrice")
y = data["HousePrice"]

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

# Fit ML model

In [4]:
model = LinearRegression()
model.fit(X_train, y_train)

LinearRegression()

# Select background data and data sample(s)

In [5]:
n_background = 100
n_samples = 3

X_background = X_train.sample(n=n_background) # if n too large shap library will alter background data so it wont equal our calculated one
x = X_test.iloc[0:n_samples]

# Configure and compute Shapley Values

In [6]:
%run shap_computation.py

In [7]:
exact_explainer = exact_SHAP(model, X_background)
exact_shap_values = exact_explainer.explain(x)

In [8]:
for i, shap_value in enumerate(exact_shap_values):
    print(f"SHAP values for row {i}: {shap_value} \n")

SHAP values for row 0: [-33.90378901  11.69356907   7.75367792  -4.41582742   0.0914193
   0.16947325  51.71316441 -63.59066591] 

SHAP values for row 1: [ 56.80513479   9.96247077  -0.67111083  -3.52576933  -0.58941432
   0.28611392 -98.27937665 131.90753059] 

SHAP values for row 2: [ 88.65069934  -1.28966824  -8.29769358  -8.85817623   0.18036431
   0.20458965  61.23045763 -50.45127592] 



## Compute and verify property Efficiency for sample $x_0$

We compute the expected prediction over the background dataset,
$$
E[f(X)] = \frac{1}{|B|}\sum_{x' \in B} f(x'),
$$
and add the sum of all SHAP values. The efficiency property states that this must reconstruct the model prediction for the observation $x$:
$$
E[f(X)] + \sum_{i=1}^{n}\phi_i = f(x).
$$
Thus, the expected prediction represents the base value, while the SHAP values represent the individual feature contributions that collectively account for the difference between the base value and the actual prediction.

In [31]:
expected_value = exact_explainer.expected_value
f_x = model.predict(x.iloc[[0]])

print(
    f"{'Expected value E[f(x)]':<45}: {expected_value:.6f}\n"
    f"{'SHAP values for x_0':<45}: {exact_shap_values[0]}\n"
    f"{'Sum of SHAP values for x_0':<45}: {sum(exact_shap_values[0]):.6f}\n"
    f"{'Model prediction f(x) for x_0':<45}: {f_x[0]:.6f}\n"
    f"{'Expected value + summed SHAP values':<45}: "
    f"{expected_value + sum(exact_shap_values[0]):.6f}"
)

Expected value E[f(x)]                       : 193.798421
SHAP values for x_0                          : [-2.96770543e+01  1.17628130e+01  3.27692379e+00  2.57599480e+00
 -3.63057357e-03 -7.45142395e-02  6.65449142e+01 -8.08351199e+01]
Sum of SHAP values for x_0                   : -26.429673
Model prediction f(x) for x_0                : 167.368747
Expected value + summed SHAP values          : 167.368747


## Test value computation efficiency

In [66]:
print("Runtime")
print("-" * 45)

start_t = perf_counter()
exact_explainer = exact_SHAP(model, X_background)
exact_shap_values = exact_explainer.explain(x)
print(f"{'Simple Exact Explainer':<30}: {perf_counter() - start_t:.3f}s")

start_t = perf_counter()
optimized_exact_explainer = exact_SHAP_optimized(model, X_background)
optimized_exact_shap_values = optimized_exact_explainer.explain(x)
print(f"{'Optimized Exact Explainer':<30}: {perf_counter() - start_t:.3f}s")

start_t = perf_counter()
shap_explainer = shap.ExactExplainer(model.predict, X_background)
shap_values = shap_explainer(x)
print(f"{'SHAP Library':<30}: {perf_counter() - start_t:.3f}s")


results = []

for i, (pred_a, pred_b, pred_c) in enumerate(zip(exact_shap_values, optimized_exact_shap_values, shap_values.values)):
    results.append({
        "Sample": f"x_{i}",
        "Simple Exact": sum(pred_a),
        "Optimized Exact": sum(pred_b),
        "SHAP Library": sum(pred_c),
    })

print("\n \nSHAP Summed Value Comparison")
print("-" * 60)

pd.DataFrame(results).set_index("Sample").round(6)

Runtime
---------------------------------------------
Simple Exact Explainer        : 3.354s
Optimized Exact Explainer     : 0.434s
SHAP Library                  : 0.008s

 
SHAP Summed Value Comparison
------------------------------------------------------------


,Simple Exact,Optimized Exact,SHAP Library
Sample,,,
x_0,-25.089764,-25.089764,-25.089764
x_1,101.294793,101.294793,101.294793
x_2,86.768511,86.768511,86.768511


# Limits of Exact Explainer

The computational cost of the Exact Explainer depends on both the number of samples to be explained and the size of the background dataset. Let $\mathrm{m}$ denote the number of samples to explain, $\mathrm{B}$ the number of background samples and $\mathrm{p}$ the number of features.

For a fixed number of features and background samples, the computational cost scales approximately linearly with the number of samples to explain:

$$
T(m) = O(m).
$$

Thus, if $\mathrm{m}$ samples require $\mathrm{x}$ seconds, then $\mathrm{10m}$ samples will require approximately $\mathrm{(10m/m)x}$ seconds, assuming all other factors remain constant.

The background dataset affects the cost of each coalition evaluation because the model prediction is computed over all $\mathrm{B}$ background samples. In principle, this introduces an approximately linear dependence on $\mathrm{B}$:

$$
T(B) = O(B).
$$

In practice, however, this dependence may be less noticeable for moderate background sizes. Modern machine-learning libraries perform predictions in batches using vectorized and highly optimized implementations, so increasing $\mathrm{B}$ does not necessarily result in a proportional increase in wall-clock time for small or moderate values of $\mathrm{B}$. The SHAP library also commonly uses a bounded background sample size of $100$, which limits this source of computational growth.

The dominant limitation of the Exact Explainer is instead the exponential dependence on the number of features. For a feature $\mathrm{i}$, its exact Shapley value requires evaluating the marginal contribution of $\mathrm{i}$ for every subset $S \subseteq N \setminus \{i\}$. With $\mathrm{p}$ features, there are

$$
2^{p-1}
$$

such subsets for each feature. Consequently, computing the exact Shapley values for all $\mathrm{p}$ features requires

$$
p2^{p-1}
$$

coalition evaluations.

Therefore, ignoring the cost of an individual model evaluation, the overall computational complexity can be approximated as

$$
O\left(mp2^{p-1}B\right).
$$

This exponential dependence on $\mathrm{p}$ is the fundamental scalability limitation of exact Shapley-value computation. While increasing the number of explained samples or background samples primarily introduces linear scaling, increasing the number of features causes the number of required coalition evaluations to grow exponentially.


In [67]:
X_background = X_train.sample(n=n_background) 
x = X_test.iloc[0:n_samples*10]

start_t = perf_counter()
shap_explainer = shap.ExactExplainer(model.predict, X_background)
shap_values = shap_explainer(x)
print(f"{'SHAP Library':<30}: {perf_counter() - start_t:.3f}s")

SHAP Library                  : 0.075s


In [68]:
X_background = X_train.sample(n=n_background*5) 
x = X_test.iloc[0:n_samples*10]

start_t = perf_counter()
optimized_exact_explainer = exact_SHAP_optimized(model, X_background)
optimized_exact_shap_values = optimized_exact_explainer.explain(x)
print(f"{'Optimized Exact Explainer':<30}: {perf_counter() - start_t:.3f}s")

Optimized Exact Explainer     : 4.239s


# Dataset with more features

The Ames Housing dataset contains 2,930 residential property sales and 82 columns, including numerous property characteristics that make it well suited for demonstrating the computational limitations of exact SHAP value computation.


In [7]:
ames_housing = pd.read_csv("../data/ames_housing_dataset.csv")
ames_housing.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


### 'Cleaning' the dataset

As we are not concerned with model accuracy or other performance metrics, we drop all non-numeric columns to allow the LinearRegression model to work with the data.


In [20]:
numeric_features = [col for col, dtype in zip(ames_housing.columns, ames_housing.dtypes) if dtype in ("int64", "float64")]
target = "SalePrice"

# Limitations of Exact Explainer

The code below demonstrates the computational limitations of calculating every possible feature combination and, consequently, the marginal contribution of all features. Even though our test instance `x` and background data `X_background` are relatively small, the time required for the exact calculation grows exponentially with the number of features.

The Exact Explainer evaluates $2^p$ possible feature masks, where $p$ is the number of features. This means that adding just one additional feature doubles the number of required evaluations. In this example, we calculate 

$2^5 = 32$ (~0.005s), \
$2^{10} = 1,024$ (~0.04s), \
$2^{15} = 32,768$ (~0.93s), \
$2^{20} = 1,048,576$ (~24.08s) and \
$2^{21} = 2,097,152$ (~132.94s) 

masked evaluations.

Hence, the number of evaluations quickly becomes impractical as the number of features increases. By default, SHAP's ExactExplainer is limited to 100,000 evaluations. We can calculate the corresponding number of features:

$$
2^p = 100,000
$$

$$
p = \lfloor \log_2(100,000) \rfloor = 16
$$


In [57]:
prev_t = None
for n_features in (5, 10, 15, 20, 21):
    X = ames_housing[numeric_features[:n_features]]
    y = ames_housing[[target]]

    X = X.dropna()
    y = y.loc[X.index]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

    model = LinearRegression()
    model.fit(X_train, y_train)

    X_background = X_train.sample(n=100)
    x = X_test[0:3]

    start_t = perf_counter()
    explainer = shap.ExactExplainer(model.predict, X_background)
    shap_values = explainer(x, max_evals=2_097_152, silent=True)
    end_t = perf_counter() - start_t
    print(f"{n_features} features: {end_t:.3f}s - {end_t / prev_t if prev_t else 0:.2f}x slower")
    prev_t = end_t

5 features: 0.005s - 0.00x slower
10 features: 0.040s - 7.55x slower
15 features: 0.930s - 23.36x slower
20 features: 24.081s - 25.90x slower
21 features: 132.941s - 5.52x slower


# TODO


---

- potentially: dig deeper into tree models and other ways of calculating shap values rather than calculating all subsets
